In [1]:
from scipy.optimize import least_squares

In [2]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/FUNCTIONS/load_functions"

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


fortran mie routines unavailable


/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:47: UserWarning: could not import compiled mie code - mie calculation will be slow
  warnings.warn(


In [3]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/constants.py"

# Schechter Function

I will define the function as:
$\frac{dN}{dS} = \phi \left(\frac{S}{S_0}\right)^\alpha -\exp\left(\frac{S}{S_0}\right)$,

where $\phi$ may be $N_0$ or $\frac{N_0}{S_0}$ depending on the papers definition

In [4]:
def Schechter(S, S0, alpha, phi, spacing = "linear"):
    
    dN_dS = phi * (S/S0)**alpha * np.exp(-S/S0)

    
    if spacing == "log":
        dN_dS = phi * (S/S0)**alpha * np.exp(-S/S0) * np.log(10) * S
    
    return dN_dS


$\frac{dN}{d \log(S)} = \ln(10) * S * \frac{dN}{dS}$

or 

$\frac{dN}{dS} = \frac{dN}{d \log(S)} / ( \ln(10) * S)$

In [5]:
def Find_N0(dN_dS, S, S0, alpha, spacing = "linear"):   
    
    if spacing == "log":
        dN_dS = dN_dS / (np.log(10) * S)

    
    N0 = dN_dS / ((S/S0)**alpha * np.exp(-S/S0))
    
    return N0


# Chen

In [11]:
df_Chen = pd.DataFrame({
    "S": [0.25, 0.42, 0.69, 1.15, 1.90, 3.14, 5.19],
    "N Given": np.array([8, 15, 16, 14, 14, 12, 9]),
    "dN/dS Given": [73860, 28600,  10230,  2780, 1280, 530, 220]# Units of mJy^-1 deg^-2
})


df_Chen[""] = ["|" for S in df_Chen["S"]]

df_Chen["N0"] = Find_N0(df_Chen["dN/dS Given"], df_Chen["S"], 3.9, -1.7)

df_Chen["Avg. N0"] = [np.mean(df_Chen["N0"])] + [" "] * (len(df_Chen) - 1)

df_Chen[" "] = ["|" for S in df_Chen["S"]]


# Input: (S, S0, alpha, phi, spacing = "linear")
df_Chen["dN/dS .C"] = Schechter(df_Chen["S"], 3.9, -1.7, 2.2)

df_Chen["dN/dS.diff"] = np.abs(df_Chen["dN/dS .C"] - df_Chen["dN/dS Given"])




df_Chen

,S,N Given,dN/dS Given,,N0,Avg. N0,,dN/dS .C,dN/dS.diff
0,0.25,8,73860,|,737.796985,765.200678,|,220.239447,73639.760553
1,0.42,15,28600,|,720.858742,,|,87.284785,28512.715215
2,0.69,16,10230,|,642.607769,,|,35.022919,10194.977081
3,1.15,14,2780,|,468.255067,,|,13.061257,2766.938743
4,1.90,14,1280,|,613.565063,,|,4.589570,1275.410430
5,3.14,12,530,|,820.180002,,|,1.421639,528.578361
6,5.19,9,220,|,1353.141117,,|,0.357686,219.642314


# Carniani

In [7]:
df_Carniani = pd.DataFrame({
    "S": [0.03, 0.04, 0.06, 0.08, 0.12, 0.22, 0.34],
    "N": np.array([0, 0, 2, 5, 7, 5, 7]),
    "dN/d log(S)": np.array([70, 30, 11, 10, 7, 3, 4]) * 1e4

})


df_Carniani[""] = ["|" for S in df_Carniani["S"]]

df_Carniani["N0"] = Find_N0(df_Carniani["dN/d log(S)"], df_Carniani["S"], 1.7, -2.08, spacing = "log")

df_Carniani["Avg. N0"] = [np.mean(df_Carniani["N0"])] + [" "] * (len(df_Carniani) - 1)

df_Carniani[" "] = ["|" for S in df_Carniani["S"]]



# Input: (S, S0, alpha, phi, spacing = "linear")
df_Carniani["dN/d log(S) .C"] = Schechter(df_Carniani["S"], 1.7, -2.08, 1800, spacing = "log")/ 1.7

df_Carniani["dN/d log(S) .diff"] = np.abs(df_Carniani["dN/d log(S) .C"] - df_Carniani["dN/d log(S)"])

df_Carniani

,S,N,dN/d log(S),,N0,Avg. N0,,dN/d log(S) .C,dN/d log(S) .diff
0,0.03,0,700000.0,|,2325.431603,1387.886033,|,318726.411813,381273.588187
1,0.04,0,300000.0,|,1367.776992,,|,232236.001018,67763.998982
2,0.06,2,110000.0,|,786.275296,,|,148129.527637,38129.527637
3,0.08,5,100000.0,|,986.790883,,|,107299.687112,7299.687112
4,0.12,7,70000.0,|,1095.772562,,|,67639.626701,2360.373299
5,0.22,5,30000.0,|,958.497799,,|,33140.092654,3140.092654
6,0.34,7,40000.0,|,2194.657093,,|,19298.204405,20701.795595
